In [2]:
import numpy as np
from xgboost import XGBRegressor
import joblib

In [3]:
X_train, X_test, y_train, y_test, X, y = joblib.load("../Data/preprocessing_all_info.joblib")

In [9]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from skopt import BayesSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import loguniform, randint
from skopt.space import Real, Integer

In [24]:
params_grid = {"n_estimators" : [250 * i for i in range(1, 4)], "max_depth" : [3, 5, 6], "learning_rate" : [0.1, 0.01]}
params_space_random = {"n_estimators" : randint(250, 1000), "max_depth" : randint(3, 10), "learning_rate" : loguniform(1e-3, 1e-1)}
params_space_bayes = {"n_estimators" : Integer(250, 1000), "max_depth" : Integer(3, 10), "learning_rate" : Real(1e-3, 1e-1, prior="log-uniform")}

In [25]:
base_model = XGBRegressor(subsample=0.7, random_state=101)

In [27]:
grid_model = GridSearchCV(
    estimator=base_model,
    param_grid=params_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=0,
)

grid_model.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,
                                    learning_rate=None, max_bin=None,
                                    max_cat_threshold=None,
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.1, 0.01], 'max_depth': [3, 5, 6],
                         'n_estimators': [250, 500, 750]},
             scoring='r2')

In [28]:
y_pred = grid_model.predict(X_test)

MSE = mean_squared_error(y_pred, y_test)
MAE = mean_absolute_error(y_pred, y_test)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_true=y_test, y_pred=y_pred)

print(f"MAE:{MAE}, MSE:{MSE}, RMSE:{RMSE}, R^2:{R2}")

MAE:0.652958191274493, MSE:0.8099174864561978, RMSE:0.8999541579748369, R^2:0.7372129680746239


In [29]:
random_model = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=params_space_random,
    n_iter=15, 
    cv=3,
    scoring='r2',
    random_state=101,
    n_jobs=-1
)

random_model.fit(X_train, y_train)

RandomizedSearchCV(cv=3,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                   n_iter=15, n_jobs=-1,
                   param_distributions={'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001D3AC63B890>,
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000001D3AC75C2F0>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000001D3AC741630>},
                   random_state=101, scoring='r2')

In [30]:
y_pred = random_model.predict(X_test)

MSE = mean_squared_error(y_pred, y_test)
MAE = mean_absolute_error(y_pred, y_test)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_true=y_test, y_pred=y_pred)

print(f"MAE:{MAE}, MSE:{MSE}, RMSE:{RMSE}, R^2:{R2}")

MAE:0.6503832676891014, MSE:0.807477947043218, RMSE:0.8985977671034009, R^2:0.738004504659921


In [31]:
bayes_model = BayesSearchCV(
    estimator=base_model,
    search_spaces=params_space_bayes,
    n_iter=15,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=101
)

bayes_model.fit(X_train, y_train)

BayesSearchCV(cv=3,
              estimator=XGBRegressor(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=Non...
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
              n_iter=15, n_jobs=-1, random_state=101, scoring='r2',
              search_spaces={'learning_rate': Real(low=0.001, high=0.1, prior='log-uniform', transform='normalize'),
                             'max_depth': Integer(low=3, high=10, prior='uniform', transform='normalize'),
                             'n_estimators': Integer(low=250, high=1000, prior='uniform', transform='normalize')})

In [33]:
y_pred = bayes_model.predict(X_test)

MSE = mean_squared_error(y_pred, y_test)
MAE = mean_absolute_error(y_pred, y_test)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_true=y_test, y_pred=y_pred)

print(f"MAE:{MAE}, MSE:{MSE}, RMSE:{RMSE}, R^2:{R2}")

MAE:0.6488201963777263, MSE:0.7999072166562669, RMSE:0.8943753220300004, R^2:0.7404609150982228
